# SignSpeak Universal Prototype — Part 1: Dataset Acquisition & Setup

**Version:** 1.0 (Phase 0 Prototype Notebook)
**Target Hardware:** NVIDIA GeForce RTX 4050 (6 GB VRAM)
**Allocated Disk Budget:** ~4.5 GB for raw datasets (out of 58 GB available drive space)

---

### Objectives of Notebook 1:
1. **Environment Verification:** Validate Python environment, CUDA GPU access (RTX 4050), and directory structure.
2. **Dataset 1 (INCLUDE - Indian Sign Language):** Download & extract 263 ISL isolated sign classes (~2.0 GB).
3. **Dataset 2 (WLASL-100 - American Sign Language):** Download metadata & video assets for top 100 ASL isolated sign classes (~1.5 - 2.5 GB).
4. **Data Integrity Audit:** Inspect MP4 files, extract sample frame properties, and verify dataset balance.

## Step 1: Environment & Directory Initialization

In [6]:
import os
import sys
import shutil
import json
import urllib.request
import zipfile
from pathlib import Path
import torch
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm

# 1. Setup Base Project Paths
BASE_DIR = Path(r"d:\finalspeak")
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
INCLUDE_DIR = RAW_DIR / "include"
WLASL_DIR = RAW_DIR / "wlasl"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = BASE_DIR / "models"

for path in [RAW_DIR, INCLUDE_DIR, WLASL_DIR, PROCESSED_DIR, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)
    print(f"[OK] Directory ready: {path}")

# 2. Validate GPU Availability
print("\n--- System Diagnostics ---")
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Target GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Available VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[!] Warning: CUDA GPU not detected. PyTorch will use CPU.")

[OK] Directory ready: d:\finalspeak\data\raw
[OK] Directory ready: d:\finalspeak\data\raw\include
[OK] Directory ready: d:\finalspeak\data\raw\wlasl
[OK] Directory ready: d:\finalspeak\data\processed
[OK] Directory ready: d:\finalspeak\models

--- System Diagnostics ---
Python Version: 3.12.4
PyTorch Version: 2.5.1+cu121
CUDA Available: True
Target GPU Device: NVIDIA GeForce RTX 4050 Laptop GPU
Available VRAM: 6.00 GB


## Step 2: Download & Extract Dataset 1 — INCLUDE (Indian Sign Language)

In [7]:
def download_file(url, destination_path):
    """Helper to download files with progress bar."""
    class DownloadProgressBar(tqdm):
        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None:
                self.total = tsize
            self.update(b * bsize - self.n)

    print(f"[Downloading] {url} -> {destination_path.name}")
    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=destination_path.name) as t:
        urllib.request.urlretrieve(url, filename=destination_path, reporthook=t.update_to)
    print(f"[OK] Download finished: {destination_path.name}")

# INCLUDE dataset direct Zenodo / GitHub release mirrors
INCLUDE_ZIP = RAW_DIR / "include_dataset.zip"
# Zenodo Direct Download for INCLUDE ISL Dataset (4010759 / IIT Madras)
INCLUDE_URL = "https://zenodo.org/record/4010759/files/INCLUDE.zip?download=1"

if not any(INCLUDE_DIR.iterdir()):
    if not INCLUDE_ZIP.exists():
        try:
            print("Attempting direct download of INCLUDE ISL Dataset (~2.0 GB)...")
            download_file(INCLUDE_URL, INCLUDE_ZIP)
        except Exception as e:
            print(f"[!] Zenodo download automated trigger notice: {e}")
            print("If direct link requires mirror download, run curl/kaggle/drive sync or manual extract.")
    
    if INCLUDE_ZIP.exists():
        print(f"Extracting {INCLUDE_ZIP.name} into {INCLUDE_DIR}...")
        with zipfile.ZipFile(INCLUDE_ZIP, 'r') as zip_ref:
            zip_ref.extractall(INCLUDE_DIR)
        print("[OK] Extraction Complete!")
else:
    print("[OK] INCLUDE dataset directory is already populated.")

[OK] INCLUDE dataset directory is already populated.


## Step 3: Download & Extract Dataset 2 — WLASL-100 (American Sign Language)

In [8]:
# WLASL Metadata & Json Index
WLASL_JSON_PATH = WLASL_DIR / "WLASL_v0.3.json"
WLASL_JSON_URL = "https://raw.githubusercontent.com/dxli94/WLASL/master/start_kit/WLASL_v0.3.json"

# Download WLASL Index
if not WLASL_JSON_PATH.exists():
    print("Downloading WLASL v0.3 metadata index...")
    download_file(WLASL_JSON_URL, WLASL_JSON_PATH)

with open(WLASL_JSON_PATH, 'r', encoding='utf-8') as f:
    wlasl_data = json.load(f)

print(f"[OK] Loaded WLASL Master Metadata. Total Signs in Index: {len(wlasl_data)}")

# Filter Top 100 Most Frequent ASL Signs for Phase 0 Prototype
wlasl_100 = wlasl_data[:100]
wlasl_100_words = [item['gloss'] for item in wlasl_100]
total_videos_100 = sum(len(item['instances']) for item in wlasl_100)

print(f"--- WLASL-100 Prototype Subset Summary ---")
print(f"Target Vocabulary: {len(wlasl_100_words)} Glosses")
print(f"First 10 Glosses: {wlasl_100_words[:10]}")
print(f"Total Video Instances in Subset: {total_videos_100}")

# Save WLASL-100 Filtered Index
WLASL_100_JSON_PATH = WLASL_DIR / "wlasl_100_index.json"
with open(WLASL_100_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(wlasl_100, f, indent=2)
print(f"[OK] WLASL-100 Index saved to {WLASL_100_JSON_PATH}")

[OK] Loaded WLASL Master Metadata. Total Signs in Index: 2000
--- WLASL-100 Prototype Subset Summary ---
Target Vocabulary: 100 Glosses
First 10 Glosses: ['book', 'drink', 'computer', 'before', 'chair', 'go', 'clothes', 'who', 'candy', 'cousin']
Total Video Instances in Subset: 2038
[OK] WLASL-100 Index saved to d:\finalspeak\data\raw\wlasl\wlasl_100_index.json


## Step 4: Dataset Video Auditing & Integrity Verification

In [9]:
def audit_video_directory(dir_path):
    """Scans video files in directory and extracts metadata."""
    video_files = list(Path(dir_path).rglob("*.mp4")) + list(Path(dir_path).rglob("*.avi"))
    print(f"Scanning directory: {dir_path}")
    print(f"Total video files found: {len(video_files)}")
    
    if not video_files:
        return pd.DataFrame()
        
    sample_records = []
    for v_path in tqdm(video_files[:20], desc="Auditing Video Headers"):
        cap = cv2.VideoCapture(str(v_path))
        if cap.isOpened():
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = cap.get(cv2.CAP_PROP_FPS)
            frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            duration = frames / fps if fps > 0 else 0
            sample_records.append({
                "Filename": v_path.name,
                "Resolution": f"{width}x{height}",
                "FPS": round(fps, 2),
                "Frames": frames,
                "Duration (s)": round(duration, 2)
            })
        cap.release()
        
    df = pd.DataFrame(sample_records)
    return df

print("--- Auditing Raw Video Datasets ---")
include_df = audit_video_directory(INCLUDE_DIR)
if not include_df.empty:
    print("\nSample INCLUDE Video Metadata:")
    print(include_df.head(5))

wlasl_df = audit_video_directory(WLASL_DIR)
if not wlasl_df.empty:
    print("\nSample WLASL Video Metadata:")
    print(wlasl_df.head(5))

--- Auditing Raw Video Datasets ---
Scanning directory: d:\finalspeak\data\raw\include
Total video files found: 0
Scanning directory: d:\finalspeak\data\raw\wlasl
Total video files found: 0


## Step 5: Disk Allocation Summary & Next Steps

In [10]:
def get_dir_size_gb(path):
    """Calculates directory size in GB."""
    total_bytes = sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file())
    return total_bytes / (1024 ** 3)

raw_size_gb = get_dir_size_gb(RAW_DIR)
print("=====================================================")
print("          DATASET ACQUISITION SUMMARY STATUS          ")
print("=====================================================")
print(f"Raw Data Directory Path: {RAW_DIR}")
print(f"Current Disk Footprint: {raw_size_gb:.2f} GB")
print(f"Budget Limit Allocated: 4.50 GB")
print(f"Remaining Headroom on Drive: {58.0 - raw_size_gb:.2f} GB")
print("=====================================================")
print("Next Notebook: part_2.ipynb (MediaPipe Holistic Feature Extraction & .npz Landmark Caching)")

          DATASET ACQUISITION SUMMARY STATUS          
Raw Data Directory Path: d:\finalspeak\data\raw
Current Disk Footprint: 3.49 GB
Budget Limit Allocated: 4.50 GB
Remaining Headroom on Drive: 54.51 GB
Next Notebook: part_2.ipynb (MediaPipe Holistic Feature Extraction & .npz Landmark Caching)
